# Overview 

The supply chain runs from mines to manufacturers, with markets facilitating the trade between each stage.

The initial dataset used to initialise the mines is from the below source:

https://github.com/johannahansel/lithium-datasets

Eventually, I would like to pair this dataset with the following dataset:

https://ecl.earthchem.org/view.php?id=3329
https://ui.adsabs.harvard.edu/abs/2024ieda.data..140B/abstract


This provides a snapshot of all the different deposits around the world. I could match on the longitude and latitude to add more operational detail. This would allow me to add more detail ore grade and project type information.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

working_directory = Path.cwd().resolve()

model_root = next(
    parent / "02_Model"
    for parent in (working_directory, *working_directory.parents)
    if (parent / "02_Model" / "v3" / "src").is_dir()
)

if str(model_root) not in sys.path:
    sys.path.insert(0, str(model_root))

from copy import deepcopy
from math import fsum, isclose

from v3.src.mines import Mine
from v3.src.model import Model

# Model boundaries

The model runs through the three supply chain stages:

1. Mining
2. Refining
3. Manufacturing

This inherently aggregates a lot of steps, while also making some relatively large assumptions around what occurs at which facility. In reality, there a lot of stages in the supply chain. Sometimes these occur onsite for some projects and offsite in a new location for others.

To make the model both tractable and generalisable across different minerals / production routes, the decision was made that we must aggregate to three supply chain stages. 

## Example: Lithium

To illustrate this, we take the example of lithium. This is a sensible place to start as it is also the first material that will be modelled.

1.  **Mining**: Extraction and initial concentration, producing spodumene concentrate or concentrated lithium solution.
2.  **Refining**: Chemical conversion and purification, initially represented as producing one standard lithium carbonate product.
3.  **Manufacturing**: Transformation of refined chemicals into materials, components and final products, aggregated according to the model's scope.

This requires that we compare costs for the different routes. As each mine type supplies a different intermediate, we need to calculate what its lithium content would cost after transport and refining into lithium carbonate. Each entry then represents a mine plus its assumed downstream processing route.

This is needed to produce the cost curves which determine whether it is profitable to open a mine.

An example table below shows this calculation.

| Cost component | Hard-rock route | Brine route |
|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|--|-------------------------------------------------------|
| Mining/concentration cost per tonne of LCE in intermediate | \$3,000 | \$2,000 |
| Transport per tonne of LCE in intermediate | \$200 | \$100 |
| Refining recovery | 80% | 90% |
| Refining cost per tonne of final carbonate | \$2,000 | \$1,000 |
| Combined operating cost per tonne of final carbonate | \$6,000 | \$3,333 |

This is illustrative. Costs differ greatly for each site based on the grade and number of by-products.

### Assumptions

1. Extraction and concentration occur at one modelled site.
2. Mining capacity limits annual extraction of ore or raw brine. On-site concentration/processing has sufficient capacity to process this material within the model period. Stockpiling and processing delays are initially omitted. 

### Lithium routes

![Lithium supply chain](../../../FIGURES/draw_io/outputs/Proposal_diagrams-Li_SC.drawio.png)

# Understanding the supply chain connections: the market

There is no global lithium price. This makes producing these curves more complicated. Prices instead refer to a particular lithium product, quality and delivery location. This means that the price differs for every supplier-purchaser relationship, varying by quality, distance, and supply-chain maturity.

The cost-curve we develop is therefore highly illustrative, as it collapses these complex mechanisms into a single figure.

There are several lithium benchmark prices out there:

1.  **Battery-grade lithium carbonate:** lithium carbonate meeting a defined specification, commonly \>99.5% purity. This is an output of the **REFINING** stage of the illustrative supply chain.
2.  **Battery-grade lithium hydroxide monohydrate:**\* A different refined lithium chemical, with its own specification and price. This is also an output of the **REFINING** stage.
3.  **Spodumene concentrate (often SC6):** Concentrate containing approximately 6% lithium oxide. This is an output of the **MINING** stage, but only for hard-rock mines.

For the first version of my model, one standard battery-grade lithium carbonate product is the only refining output. Its price represents the market between refining and manufacturing.

### The market

The mining outputs become that product through physical conversion in refining. Therefore the markets connects as mines sell intermediates, refiners convert intermediates, refiners sell carbonate, manufacturers purchase carbonate.

This then leads to two price questions:

1.  What will a refner pay for the mine's intermediate?
2.  What will a manufacturer pay for the refiner's carbonate?

The carbonate price influences the first price because it determines what a refiner can afford to pay for feedstock after covering conversion, losses and transport.

A useful starting calculation is the refiner's maximum affordable purchase price at the mine gate:

$$ b_{ij} = \eta _{ij} (P_j - c_j ) - T_{ij} $$

where: - $P_j$: expected carbonate selling price at refiner $j$'s gate, per tonne of carbonate. - $c_j$: conversion operating cost per tonne of carbonate - $n_{ij}$: tonnes of carbonate recovered per tonne of LCE in mine i's intermediate - $T_{ij}$: transport cost from mine $i$ to refiner $j$, per tonne of intermediate LCE.

For an invented example, if carbonate sells for \$10,000/t, conversion costs \$3,000/t, recovery is 90%, and transport costs \$200/t of intermediate LCE:

$$ b_{ij} = 0.9 (10,000 - 3,000) - 200 = \$ 6,100$$

That is an operating break-even ceiling for the refiner's feedstock purchase. A desired margin would lower its bid.

It is not automatically the transaction price: the market mechanism must determine the price paid between the mine's offer and the refiner's bid.

#### Transport costs are simplified initially

In the first version, we use one representative global refiner. This is considered a 'worldwide' refiner existing at all locations, not one. Transport costs are therefore homogeneous (or according to a random distribution later). In later versions, I will calculate the transport cost for each candidate connection, then use these costs when deciding which connections trade.

| Candidate connection | Transport estimate | Consequence |
|--------------------|---------------|-------------------------------------|
| Mine A -\> Refiner X | $T_{AX}$ | Determines X's delivered cost and affordable bid |
| Mine A -\> Refiner Y | $T_{AY}$ | Determines Y's delivered cost and affordable bid |
| Mine B -\> Refiner X | $T_{BX}$ | Provides an alternative source for X |

The market then allocates quantities subject to offers, bids, mine supply and refinery capacity. Only selected connections generate shipments and transport expenditure.

Initially these estimates could come from a simple country-to-country cost table. Later they could depend on distance, mode, ports and material form. Freight costs apply to the physical cargo mass.

### What this means for the single cost chart

There are two valid uses, at different stages of the work:

- **Before trade allocation:** an illustrative curve using explicitly assumed downstream routes. Its transport assumptions are scenarios, not established mine attributes.

- **After trade allocation:** a curve based on selected routes, their actual allocated quantities, transport costs and refining recoveries.

For a candidate mine-refiner route, its production cost remains:

$$C_{ij} = \frac {c_{i} + T_{ij}}{\eta _{ij}} + c_{j}$$

Here $c_i$ is the mine's production cost per tonne of intermediate LCE. This is a whole-chain cost calculation; the actual refiner's accounts instead use the purchase price it pays for the feedstock.

### Current model

For the intermediate work, I will keep mine-gate costs and capacities independent of destination, and make any combined carbonate curve a clearly labelled scenario calculation.

## How the market operates

There are two options I will consider for where the market price comes from:

#### Option 1: Exogenous price

A supplied price series or scenario. This can be appropriate for testing costs and production responses.

#### Option 2: Competitve market clearing

Aggregate supply and demand determine price. Modelling feedback between demand, shortages and production.

For the eventual ABM, competitive clearing is the more useful baseline, however an illustrative exogenous price series is used initially to delay designing the market mechanism.

### Initial market clearing

With the representative global refining sector, a simple competitive calculation could rank routes by their combined operating cost and accept enough supply to meet demand. When demand intersects available supply, the marginal accepted route can determine the carbonate price under a stated auction rule. I would need a separate rule for insufficient supply and partially used marginal capacity.

# Single global refiner

The representative global refiner assumption is made to make the initial calculations much more tractable.

- It is a **pooled refining sector**, not one monopolistic firm. I implicitly assume perfect competition initially.
- I however preserve two processing 'recipes' within that pool: different conversion costs and recoveries for the two intermediates.
- Delivered prices can differ by product, specification, contract and location. Prices can differ for every supplier-purchaser relationship, but the transport costs are the same.
- The illustrative cost-curve is not what mine investment decisions are made based on. The cost-curve is intended to be illustrative, not what the mining agents make decisions on. It instead plots the battery-grade lithium price against the supply quantity that it implies becomes economically feasible at that price point. 

## Calculating costs for the mines

#### Step 1: Recovered intermediate lithium per tonne of ore or brine

First, calculate recovered intermediate lithium per tonne of ore or brine:

$$ y_i = \text {contained LCE per tonne of feed}_i \times \eta _ i $$

where: - \$ \eta \_i \$ is recovery efficiency - $y_i$ is recovered LCE per tonne of feed.

For the native grade units:

$$ y _ {\text{hard rock}} = \frac {g_i}{ \rho _{brine}} \times 5.323 \times \eta _{i} ^{mine} $$

$$ y _ {\text{brine}} = \frac {g_i 10 ^{-6}}{100} \times 2.473 \times \eta _{i} ^{mine} $$

#### Step 2: Calculate the mining-stage unit cost:

$$ c_i^{mine} = \frac{a_i}{y_i}$$

where $a_i$ is extraction and expenditure per tonne of feed.

This allows us to to calculate the implied upstream cost per tonne of intermediate LCE:

| Route | Recovered intermediate LCE / ton of feed | Assumed cost per tonne of feed | Implied upstream cost per tonne of intermediate LCE |
|-----------------|-------------------|-------------------|-------------------|
| Hard rock | 0.01929 | \$125 | \$6,480 |
| Brine | 0.001464 | \$5 | \$3,416 |

: Example calculation using current median distribution

#### Step 3: Calculate the comparable carbonate cost under the pooled-refining scenario

$$C_i = \frac{c_i^{mine} + T_r}{\eta _r ^{ref}} + c_r^{ref}$$

## Notation table

| Symbol | Meaning | Units / interpretation | Datafram equivalent |
|-------|----------------------|----------------------|----------------------|
| $g_i$ | Grade of the ore or raw brine | $$ \text{Hard rock: } \% Li_2 O $$
$$\text{Brine: mg Li/litre}$$ | grade |
| $\rho$ | Raw brine density | Tonnes of brine/m\^3 | BRINE_DENSITY |
| $f_i$ | Lithium content of feed, expressed as LCE, before recovery | Tonnes contained LCE/tonne feed | Not separately calculated (this should be added) |
| $\eta _ i ^{mine}$ | Mining/concentration recovery | Fraction of feed lithium captured in the intermediate, between 0 and 1 | recovery_efficiency |
| $y_i$ | Recovered intermediate yield | Tonnes LCE in intermediate/tonne feed | Intended meaning of lce_fraction |
| $M_i$ | Feed processed during a period | Tonnes of ore or raw brine | Not yet calculated |
| $Q_i$ | Intermediate output during that period, expressed as LCE | Tonnes recovered LCE | Not yet calculated |
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |

For hard rock:

$$f_i = \frac{g_i}{100} \times 2.473$$

$$y_i = f_i \eta _i^{mine}$$

For brine:

$$f_i = \frac{g_i \times 10^{-6}}{\rho} \times 5.323$$

$$y_i = f_i \eta_i^{mine}$$

Here, $10^{-6}$ converts mg Li/litre to tonnes $Li / m^3$. Dividing by density converts the denominator to tonnes of brine. There is no /100 in this calculation.

### Example: Hard rock through the chain:

$$y_i = 0.0012 \times 2.473  \times 0.65 = 0.0192894$$

That means that one tonne of ore yields intermediate containing 0.0192894 tonnes LCE. It does not mean one tonne yields that mass of physical concentrate. Processing 1,000 tonnes of this ore gives:

$$Q_i = M_i y_i = 1,000 \times 0.0192894 = 19.2894 \text{ tonnes LCE}$$

The lithium balance, expressed in LCE units, is:

- Entering in feed: $M_if_i = 29.676$ tonnes LCE

- Recovered in intermediate: \$M_i f_i \eta\_i \^{mine} = 19.2894

- Unrecovered: $M_i f_i(1-\eta_i^{mine}) = 10.3866$

## Capacity 

Capacity is a separate quantity. If $K_i^{feed}$ is annual feed-processing capacity in tonnes/year, and $u_i$ is utilisation:

$$M_i = u_i K_i^{feed}$$

for a full operating year, subject to available stock. Annual intermediate output is then: \$ Q_i = M_i y_i\$.

### Problem: Capacity

My current capacity is not explicitly feed-processing capacity. It is calculated as reserves divided by years. Under our provisional interpretation of reserves as contained geological LCE, its units are tonnes of geological LCE/year, before recovery. We must resolve and label that basis before using it calculate throughput or recovered output.

## Calculating operating costs

Define:

- $a_i$: extraction/concentration cost, in USD per tonne of feed.

- $c_i^{mine}$: mining-stage cost, in USD per tonne of LCE recovered into intermediate

Then:

$$c_i^{mine} = \frac{a_i}{y_i}$$

For hard-rock, using the median from the illustrative figures:

$$c_i^{mine} = \frac{125}{0.0192894} \approx $6,480 /\text{tonne intermediate LCE}$$

Capacity is not required for this unit-cost calculation under the assumed proportional cost structure.

# Initialise the facilities

The first step is to create all the different facilities that will be used. This is done by loading from external data that is cleaned, but for the demonstrator, this will be created artificially below.

In [2]:
# create mine

# mine = f.Mine(asset_id="MIN_001", region = "AUS", capacity = 10, reserves =100, rom_cost = 10, composition = {"lithium" : 0.015}, recovery_rate = 0.9, coordinates = 1 )

## Demonstration

### 1. Create two illustrative mines

In [ ]:
def make_example_mine(asset_id, deposit_type, lithium_fraction):
    return Mine(
        asset_id=asset_id,
        region="illustrative",
        deposit_type=deposit_type,
        project_status="operation",
        geological_stock_t_material=200.0,
        capacity_t_material_annual=100.0,
        composition_mass_fractions={"Li": lithium_fraction},
        mining_recovery_fraction=0.65,
        opex_usd_per_t_material=5.0,
    )


hard_rock = make_example_mine(
    "hard_rock_example", "hard_rock", 0.01
)

brine = make_example_mine(
    "brine_example", "brine", 0.001
)

model = Model(mines=[hard_rock, brine])

### 2. Run the first period

In [ ]:
for mine in model.mines:
    geology = ("geological_stock", mine.input_form)
    before = deepcopy(mine.stocks)

    extracted = mine.extract(100.0)
    assert isclose(extracted, 100.0)
    assert isclose(mine.reserves, 100.0)

    product = mine.stocks[("inventory", mine.output_form)]
    assert isclose(product[mine.grade_component] / fsum(product.values()), mine.target_output_grade)

    # Every component taken from the deposit ends up in the product or the residue.
    residue = mine.stocks[("residue", mine.residue_form)]
    for component in mine.composition:
        taken = before[geology][component] - mine.stocks[geology][component]
        assert isclose(taken, product[component] + residue[component])

assert model.ledger.check_conservation()

tables = model.ledger.to_dataframes()

display(tables["events"])
display(tables["movements"])
display(tables["current_stocks"])

### 3. Advance the model once and process the remaining stocks

In [ ]:
model.period += 1

for mine in model.mines:
    mine.extract(100.0)
    assert isclose(mine.reserves, 0.0, abs_tol=1e-9)

assert model.ledger.check_conservation()

display(model.ledger.to_dataframes()["events"])